#### Chatbot Evaluation

In [43]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [44]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['2a3b96b3-a6f5-4ba0-b184-5841d2d1dcc1',
  '552938b2-a20e-4a1e-a40f-4bf06eec253b',
  'db1a4764-81b3-4d05-abe1-b4d99aea80c7',
  'bb593c8e-3663-400f-8c82-894f976e110c',
  '0dc53b4e-6a54-4230-822b-c8ab7dd4e1b4'],
 'count': 5,
 'as_of': '2026-03-04T11:43:28.977670342Z'}

#### Define Metrics (LLM As A Judge)


In [45]:
import openai
from langsmith import wrappers
 
openai_client=wrappers.wrap_openai(openai.OpenAI())

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict,outputs:dict, reference_outputs:dict)->bool:
      user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
      response=openai_client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0,
            messages=[
                  {"role":"system","content":eval_instructions},
                  {"role":"user","content":user_content}
            ]
      ).choices[0].message.content

      return response == "CORRECT"

In [46]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

#### Run Evaluations

In [47]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str, model: str = "gpt-3.5-turbo", instructions: str = default_instructions) -> str:
    return openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [48]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [49]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-3.5-turbo-chatbot"
)

View the evaluation results for experiment: 'openai-3.5-turbo-chatbot-94ef5f92' at:
https://smith.langchain.com/o/ed875a17-f1db-4f8d-8ba4-8eed0a9c26b2/datasets/cbcef292-4079-40a7-afce-29bb42c32329/compare?selectedSessions=9a1db2f1-b3fd-48ff-aa10-42c8a7bc136e




5it [00:08,  1.79s/it]


In [50]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"],model="gpt-3.5-turbo")}

In [51]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-3.5-turbo-chatbot"
)

View the evaluation results for experiment: 'openai-3.5-turbo-chatbot-6fc293c6' at:
https://smith.langchain.com/o/ed875a17-f1db-4f8d-8ba4-8eed0a9c26b2/datasets/cbcef292-4079-40a7-afce-29bb42c32329/compare?selectedSessions=08da23fa-fbd5-4e02-8a83-f0daf3f9a766




5it [00:08,  1.72s/it]


#### Evaluation For RAG

In [52]:
## RAG
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = [
    "https://en.wikipedia.org/wiki/Electric_vehicle",
    "https://www.caranddriver.com/features/a26962316/how-a-car-engine-works/",
]

# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

# Add the document chunks to the "vector store" using OpenAIEmbeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=OpenAIEmbeddings(),
)

# With langchain we can easily turn any vector store into a retrieval component:
retriever = vectorstore.as_retriever(k=6)

In [53]:
retriever.invoke("what is agents")

[Document(id='4f00bc7c-6f88-44f1-9578-f9aa6b57f005', metadata={'source': 'https://en.wikipedia.org/wiki/Electric_vehicle', 'title': 'Electric vehicle - Wikipedia', 'language': 'en'}, page_content='^ Shafie-khah, Miadreza; Heydarian-Forushani, Ehsan; Osorio, Gerardo J.; Gil, Fabio A. S.; Aghaei, Jamshid; Barani, Mostafa; Catalao, Joao P. S. (November 2016). "Optimal Behavior of Electric Vehicle Parking Lots as Demand Response Aggregation Agents". IEEE Transactions on Smart Grid. 7 (6): 2654–2665. Bibcode:2016ITSG....7.2654S. doi:10.1109/TSG.2015.2496796.\n\n^ "It\'s not just cars driving the EV revolution in emerging markets". www.schroders.com. Retrieved 12 April 2023. Intermittency from solar or wind technologies can put creating voltage and frequency variations. Batteries can charge and discharge to stabilise the grid in such instances. The batteries of electric vehicles, e-buses or electric two-wheelers, while connected to the grid, could therefore play a role in protecting a grid\'

In [54]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("openai:gpt-4o-mini")
llm

ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000014F509D3280>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000014F509D27A0>, root_client=<openai.OpenAI object at 0x0000014F55C5C520>, root_async_client=<openai.AsyncOpenAI object at 0x0000014F509D2860>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [55]:
from langsmith import traceable

## Add decorator
@traceable()
def rag_bot(question:str)->dict:
    ## Relevant context
    docs=retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are an expert automotive assistant.
    Use the following technical documents to answer the user's questions about cars.
    If the answer isn't in the documents, say you don't know.

Documents:
{docs_string}"""
    
    ## llm invoke

    ai_msg=llm.invoke([
         {"role": "system", "content": instructions},
        {"role": "user", "content": question},

    ])
    return {"answer":ai_msg.content,"documents":docs}



In [56]:
rag_bot("What is agents")

{'answer': "I don't know.",
 'documents': [Document(id='4f00bc7c-6f88-44f1-9578-f9aa6b57f005', metadata={'source': 'https://en.wikipedia.org/wiki/Electric_vehicle', 'title': 'Electric vehicle - Wikipedia', 'language': 'en'}, page_content='^ Shafie-khah, Miadreza; Heydarian-Forushani, Ehsan; Osorio, Gerardo J.; Gil, Fabio A. S.; Aghaei, Jamshid; Barani, Mostafa; Catalao, Joao P. S. (November 2016). "Optimal Behavior of Electric Vehicle Parking Lots as Demand Response Aggregation Agents". IEEE Transactions on Smart Grid. 7 (6): 2654–2665. Bibcode:2016ITSG....7.2654S. doi:10.1109/TSG.2015.2496796.\n\n^ "It\'s not just cars driving the EV revolution in emerging markets". www.schroders.com. Retrieved 12 April 2023. Intermittency from solar or wind technologies can put creating voltage and frequency variations. Batteries can charge and discharge to stabilise the grid in such instances. The batteries of electric vehicles, e-buses or electric two-wheelers, while connected to the grid, could th

#### Dataset

In [57]:
from langsmith import Client

client=Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "What is the main advantage of an electric vehicle?"},
        "outputs": {"answer": "Lower tailpipe emissions and higher energy efficiency compared to internal combustion engines."},
    },
    {
        "inputs": {"question": "How does a turbocharger work?"},
        "outputs": {"answer": "It uses exhaust gases to spin a turbine that forces more air into the combustion chamber."},
    }
]

### create the daatset and example in LAngsmith
dataset_name = "Car Knowledge Evaluation"

dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)



{'example_ids': ['f1dab37b-5aa6-4ebe-96da-610c8e31c420',
  '33a2c71c-99e4-464e-9a92-344750e9816f'],
 'count': 2,
 'as_of': '2026-03-04T11:44:00.727355116Z'}

#### Evaluators or Metrics
1. Correctness: Response vs reference answer
- Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
- Mode: Requires a ground truth (reference) answer supplied through a dataset
- Evaluator: Use LLM-as-judge to assess answer correctness.

In [58]:
from typing_extensions import Annotated,TypedDict

## Correctness Output Schema

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

## correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

from langchain_openai import ChatOpenAI

grader_llm=ChatOpenAI(model="gpt-4o-mini",temperature=0).with_structured_output(CorrectnessGrade,
                                                                         method="json_schema",strict=True)
## evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ])
    return grade["correct"]







#### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [59]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(RelevanceGrade, method="json_schema", strict=True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

#### Groundedness: Response vs retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [60]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(GroundedGrade, method="json_schema", strict=True)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

#### Retrieval Relevance: Retrieved docs vs input

In [61]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

##### Run the evaluation

In [62]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gpt-4-0125-preview"},
)
# Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-513bcde3' at:
https://smith.langchain.com/o/ed875a17-f1db-4f8d-8ba4-8eed0a9c26b2/datasets/76244274-7c7f-4dc8-a599-bc8375b20fc6/compare?selectedSessions=b3f84b67-1671-4ed0-b66f-6e970ff943e6




2it [00:27, 13.73s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,How does a turbocharger work?,I don't know.,"[page_content='most people, a car is a thing t...",None,It uses exhaust gases to spin a turbine that f...,False,False,False,False,1.845300,33a2c71c-99e4-464e-9a92-344750e9816f,019cb8a9-887f-7f51-8d9e-fcf63677a018
1,What is the main advantage of an electric vehi...,The main advantage of an electric vehicle is i...,[page_content='The carbon footprint and other ...,None,Lower tailpipe emissions and higher energy eff...,True,True,True,True,2.028092,f1dab37b-5aa6-4ebe-96da-610c8e31c420,019cb8a9-bdcf-7932-b499-54493c90f17e
